# Problem 1: Trip Duration Prediction

Bu notebook NYC bike trip verisi uzerinde trip duration tahmini icin regresyon modeli kurar.

Kullanilan ana fikirler:

- `day_of_week`: Haftanin gunu
- `hour_of_day`: Gun icindeki saat
- `is_weekend`: Hafta sonu gostergesi
- `distance_km_haversine`: Baslangic ve bitis istasyonu arasi kus ucusu mesafe
- `distance_km_manhattan`: Enlem/boylam farklarindan yaklasik Manhattan mesafesi
- Bias/intercept: Linear/Ridge model icinde `fit_intercept=True` ile modellenir

Deney tasariminda final performans metrigi icin test verisi en bastan ayrilir ve model secimi bu test verisine bakmadan yapilir.

In [ ]:
import warnings

import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)

RANDOM_STATE = 42
DATA_PATH = "s3://tt-bootcamp-shared/nyc-bike-trip/201[56]*.parquet"
SAMPLE_ROWS = 200_000

## 1. Veriyi Okuma

Veri DuckDB ile S3 uzerindeki parquet dosyalarindan okunur. Tarih kolonlari farkli formatlarda gelebilecegi icin `strptime` icinde birden fazla format denenir. Sure hedef degiskeni olarak dakika cinsinden hesaplanir.

Notebook hizli calissin diye veri setinden deterministik bir orneklem aliyoruz. Final train/test ayrimi daha sonra tarihe gore yapilacagi icin performans metrigini test verisine bakarak optimize etmiyoruz.

In [ ]:
con = duckdb.connect()
con.execute("SET extension_directory='.duckdb_extensions';")
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")
con.execute("CREATE OR REPLACE SECRET bike_s3 (TYPE s3, PROVIDER config, REGION 'eu-north-1');")

query = f"""
WITH raw AS (
    SELECT
        * EXCLUDE (tripduration, starttime, stoptime),
        strptime(starttime, ['%m/%d/%Y %H:%M', '%m/%d/%Y %H:%M:%S', '%Y-%m-%d %H:%M:%S']) AS start_at,
        strptime(stoptime,  ['%m/%d/%Y %H:%M', '%m/%d/%Y %H:%M:%S', '%Y-%m-%d %H:%M:%S']) AS stop_at
    FROM '{DATA_PATH}'
), features AS (
    SELECT
        *,
        date_diff('minute', start_at, stop_at) AS duration_min,
        extract('hour' FROM start_at) AS hour_of_day,
        dayofweek(start_at) AS day_of_week,
        CASE WHEN dayofweek(start_at) IN (0, 6) THEN 1 ELSE 0 END AS is_weekend
    FROM raw
)
SELECT *
FROM features
WHERE duration_min BETWEEN 1 AND 180
  AND "start station latitude" IS NOT NULL
  AND "start station longitude" IS NOT NULL
  AND "end station latitude" IS NOT NULL
  AND "end station longitude" IS NOT NULL
ORDER BY hash(bikeid, start_at)
LIMIT {SAMPLE_ROWS}
"""

df = con.execute(query).df()
df = df.sort_values("start_at").reset_index(drop=True)

print(df.shape)
display(df.head())

## 2. Feature Engineering

Koordinatlardan iki farkli mesafe olcusu uretiyoruz:

- Haversine distance: iki nokta arasindaki kus ucusu mesafe
- Manhattan approximation: kuzey-guney ve dogu-bati mesafelerinin toplami

Bu iki feature ayni bilgiyi farkli sekilde temsil eder. Bike rotalari genelde sokak agini takip ettigi icin Manhattan yaklasimi bazi durumlarda kus ucusu mesafeden daha anlamli olabilir.

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    radius_km = 6371.0
    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * radius_km * np.arcsin(np.sqrt(a))


def manhattan_approx_km(lat1, lon1, lat2, lon2):
    lat_km = 111.0 * np.abs(lat2 - lat1)
    mean_lat = np.radians((lat1 + lat2) / 2)
    lon_km = 111.0 * np.cos(mean_lat) * np.abs(lon2 - lon1)
    return lat_km + lon_km


df["distance_km_haversine"] = haversine_km(
    df["start station latitude"],
    df["start station longitude"],
    df["end station latitude"],
    df["end station longitude"],
)

df["distance_km_manhattan"] = manhattan_approx_km(
    df["start station latitude"],
    df["start station longitude"],
    df["end station latitude"],
    df["end station longitude"],
)

df["birth_year"] = pd.to_numeric(df["birth year"], errors="coerce")
df["age"] = df["start_at"].dt.year - df["birth_year"]
df.loc[(df["age"] < 12) | (df["age"] > 90), "age"] = np.nan

df = df[(df["distance_km_haversine"] >= 0) & (df["distance_km_haversine"] <= 30)].copy()

feature_preview = [
    "duration_min",
    "hour_of_day",
    "day_of_week",
    "is_weekend",
    "distance_km_haversine",
    "distance_km_manhattan",
    "usertype",
    "gender",
    "age",
]

display(df[feature_preview].head())
display(df[feature_preview].describe(include="all"))

## 3. Kisa Kesif

Modelden once hedef degiskenin ve mesafenin dagilimina bakiyoruz. Bu kisim model secimi icin degil, veriyi anlamak icin kullanilir.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df["duration_min"], bins=50, color="#4c78a8")
axes[0].set_title("Trip duration distribution")
axes[0].set_xlabel("Duration, min")

axes[1].scatter(df["distance_km_haversine"], df["duration_min"], s=4, alpha=0.15, color="#f58518")
axes[1].set_title("Distance vs duration")
axes[1].set_xlabel("Haversine distance, km")
axes[1].set_ylabel("Duration, min")

plt.tight_layout()
plt.show()

## 4. Experiment Design

Unbiased final metric icin test verisini zamana gore en sondaki %20 olarak ayiriyoruz. Bu, gercek hayattaki su senaryoya benzer: gecmis triplerle egitim yapip daha sonraki tripleri tahmin ediyoruz.

Kurallar:

1. Test seti sadece en sonda final performans raporu icin kullanilir.
2. Model secimi train set uzerinde yapilir.
3. Baseline model ile regresyon modeli ayni test setinde karsilastirilir.

Ana metrik olarak MAE kullanilir. Dakika cinsinden yorumlamasi kolaydir. RMSE ve R2 ek metrik olarak raporlanir.

In [ ]:
numeric_features = [
    "distance_km_haversine",
    "distance_km_manhattan",
    "hour_of_day",
    "day_of_week",
    "is_weekend",
    "age",
]

categorical_features = ["usertype", "gender"]
target = "duration_min"

model_df = df[numeric_features + categorical_features + [target, "start_at"]].dropna(subset=[target]).copy()
model_df = model_df.sort_values("start_at").reset_index(drop=True)

split_idx = int(len(model_df) * 0.80)
train_df = model_df.iloc[:split_idx].copy()
test_df = model_df.iloc[split_idx:].copy()

X_train = train_df[numeric_features + categorical_features]
y_train = train_df[target]
X_test = test_df[numeric_features + categorical_features]
y_test = test_df[target]

print("Train rows:", len(train_df), train_df["start_at"].min(), "->", train_df["start_at"].max())
print("Test rows :", len(test_df), test_df["start_at"].min(), "->", test_df["start_at"].max())

## 5. Model Kurma

Pipeline icinde numeric featurelar scale edilir, kategorik featurelar one-hot encode edilir. Ridge regresyon `fit_intercept=True` kullandigi icin bias/intercept modele dahildir.

Ayrica baseline olarak `DummyRegressor(strategy='mean')` kullanilir. Bu model her trip icin train setindeki ortalama sureyi tahmin eder.

In [ ]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocess = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features),
])

baseline = DummyRegressor(strategy="mean")

linear_model = Pipeline([
    ("preprocess", preprocess),
    ("model", LinearRegression(fit_intercept=True)),
])

ridge_model = Pipeline([
    ("preprocess", preprocess),
    ("model", Ridge(alpha=10.0, fit_intercept=True, random_state=RANDOM_STATE)),
])

tscv = TimeSeriesSplit(n_splits=5)
scoring = {
    "mae": "neg_mean_absolute_error",
    "rmse": "neg_root_mean_squared_error",
    "r2": "r2",
}

cv_results = []
for name, estimator in [("LinearRegression", linear_model), ("Ridge", ridge_model)]:
    scores = cross_validate(estimator, X_train, y_train, cv=tscv, scoring=scoring)
    cv_results.append({
        "model": name,
        "cv_mae_mean": -scores["test_mae"].mean(),
        "cv_rmse_mean": -scores["test_rmse"].mean(),
        "cv_r2_mean": scores["test_r2"].mean(),
    })

cv_results = pd.DataFrame(cv_results).sort_values("cv_mae_mean")
display(cv_results)

best_model_name = cv_results.iloc[0]["model"]
best_model = ridge_model if best_model_name == "Ridge" else linear_model
print("Selected model:", best_model_name)

## 6. Final Test Performansi

Bu noktaya kadar test setini kullanmadik. Simdi baseline ve secilen regresyon modeli ayni test seti uzerinde degerlendirilir.

In [ ]:
def regression_metrics(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": root_mean_squared_error(y_true, y_pred),
        "R2": r2_score(y_true, y_pred),
    }


baseline.fit(X_train, y_train)
best_model.fit(X_train, y_train)

baseline_pred = baseline.predict(X_test)
model_pred = best_model.predict(X_test)

test_metrics = pd.DataFrame([
    {"model": "Baseline mean", **regression_metrics(y_test, baseline_pred)},
    {"model": best_model_name, **regression_metrics(y_test, model_pred)},
])

display(test_metrics)

plt.figure(figsize=(6, 6))
plt.scatter(y_test, model_pred, s=8, alpha=0.25, color="#4c78a8")
low = min(y_test.min(), model_pred.min())
high = max(y_test.max(), model_pred.max())
plt.plot([low, high], [low, high], color="#e45756", linestyle="--")
plt.xlabel("Actual duration, min")
plt.ylabel("Predicted duration, min")
plt.title("Actual vs predicted duration")
plt.show()

## 7. Model Yorumu

Lineer modellerde intercept bias terimidir. Asagidaki hucre secilen modelin intercept degerini ve en etkili katsayilarini gosterir. Ridge secilirse katsayilar regularize edilmis olur.

In [ ]:
trained_preprocess = best_model.named_steps["preprocess"]
trained_regressor = best_model.named_steps["model"]

feature_names = trained_preprocess.get_feature_names_out()
coef_df = pd.DataFrame({
    "feature": feature_names,
    "coefficient": trained_regressor.coef_,
})
coef_df["abs_coefficient"] = coef_df["coefficient"].abs()

print("Intercept / bias:", trained_regressor.intercept_)
display(coef_df.sort_values("abs_coefficient", ascending=False).head(15))

## 8. Kisa Rapor

- Bu calismada hedef degisken olarak `duration_min` kullanildi.
- Zorunlu featurelar olan haftanin gunu, gunun saati, hafta sonu gostergesi ve istasyonlar arasi mesafe modele dahil edildi.
- Mesafe iki farkli sekilde olculdu: Haversine ve Manhattan approximation.
- Bias/intercept `fit_intercept=True` ile modele dahil edildi.
- Test seti zamana gore en sondaki %20 olarak ayrildi. Model secimi train set icindeki time-series cross-validation ile yapildi.
- Final MAE/RMSE/R2 sadece ayrilmis test setinde raporlandi. Bu nedenle test performansi model secimi sirasinda optimize edilmemis, daha tarafsiz bir tahmin olarak kullanilmistir.

## 9. Optional: Pydeck ile 10 Fictional Trip Gorsellestirme

Asagidaki kisim istasyonlardan rastgele 10 pickup/dropoff cifti secer, secilen modelle sure tahmini yapar ve yavas tripleri kirmiziya, hizli tripleri yesile yakin renklendirir.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)

start_stations = df[[
    "start station id",
    "start station name",
    "start station latitude",
    "start station longitude",
]].drop_duplicates("start station id")

end_stations = df[[
    "end station id",
    "end station name",
    "end station latitude",
    "end station longitude",
]].drop_duplicates("end station id")

pickup = start_stations.sample(10, random_state=RANDOM_STATE).reset_index(drop=True)
dropoff = end_stations.sample(10, random_state=RANDOM_STATE + 1).reset_index(drop=True)

fictional = pd.DataFrame({
    "pickup_name": pickup["start station name"],
    "dropoff_name": dropoff["end station name"],
    "pickup_lat": pickup["start station latitude"],
    "pickup_lon": pickup["start station longitude"],
    "dropoff_lat": dropoff["end station latitude"],
    "dropoff_lon": dropoff["end station longitude"],
})

fictional["distance_km_haversine"] = haversine_km(
    fictional["pickup_lat"], fictional["pickup_lon"], fictional["dropoff_lat"], fictional["dropoff_lon"]
)
fictional["distance_km_manhattan"] = manhattan_approx_km(
    fictional["pickup_lat"], fictional["pickup_lon"], fictional["dropoff_lat"], fictional["dropoff_lon"]
)

fictional["hour_of_day"] = rng.integers(6, 23, size=len(fictional))
fictional["day_of_week"] = rng.integers(0, 7, size=len(fictional))
fictional["is_weekend"] = fictional["day_of_week"].isin([0, 6]).astype(int)
fictional["age"] = train_df["age"].median()
fictional["usertype"] = train_df["usertype"].mode().iloc[0]
fictional["gender"] = train_df["gender"].mode().iloc[0]

fictional["predicted_duration_min"] = best_model.predict(fictional[numeric_features + categorical_features])
fictional["predicted_duration_min"] = fictional["predicted_duration_min"].clip(lower=1)

duration_min = fictional["predicted_duration_min"].min()
duration_max = fictional["predicted_duration_min"].max()
scaled = (fictional["predicted_duration_min"] - duration_min) / (duration_max - duration_min + 1e-9)

fictional["color"] = [
    [int(60 + 190 * s), int(180 - 130 * s), 70, 210]
    for s in scaled
]

fictional["path"] = fictional.apply(
    lambda row: [[row["pickup_lon"], row["pickup_lat"]], [row["dropoff_lon"], row["dropoff_lat"]]],
    axis=1,
)

display(fictional[["pickup_name", "dropoff_name", "predicted_duration_min", "distance_km_haversine"]])

In [ ]:
try:
    import pydeck as pdk

    view_state = pdk.ViewState(
        latitude=float(fictional[["pickup_lat", "dropoff_lat"]].to_numpy().mean()),
        longitude=float(fictional[["pickup_lon", "dropoff_lon"]].to_numpy().mean()),
        zoom=12,
        pitch=35,
    )

    route_layer = pdk.Layer(
        "PathLayer",
        data=fictional,
        get_path="path",
        get_color="color",
        get_width=35,
        width_min_pixels=3,
        pickable=True,
    )

    pickup_layer = pdk.Layer(
        "ScatterplotLayer",
        data=fictional,
        get_position="[pickup_lon, pickup_lat]",
        get_color=[30, 120, 80, 220],
        get_radius=45,
        pickable=True,
    )

    dropoff_layer = pdk.Layer(
        "ScatterplotLayer",
        data=fictional,
        get_position="[dropoff_lon, dropoff_lat]",
        get_color=[180, 40, 50, 220],
        get_radius=45,
        pickable=True,
    )

    deck = pdk.Deck(
        layers=[route_layer, pickup_layer, dropoff_layer],
        initial_view_state=view_state,
        tooltip={"text": "{pickup_name} -> {dropoff_name}\nPredicted: {predicted_duration_min} min"},
    )

    deck
except ImportError:
    print("pydeck kurulu degil. Calistirmak icin: pip install pydeck")

## Problem 2: Extending Naive Bayesian

Bu bolumde gender tahmini icin sadece bike speed bilgisini kullanan continuous Naive Bayes modeli kurulur. Speed surekli bir degisken oldugu icin $P(speed=x \mid gender=a)$ bir olasilik yogunlugu ile modellenir.

Hiz dagilimi saga kuyruklu oldugundan ham speed yerine `log_speed_kmh` de incelenir. Log donusumu dagilimi normale daha yakin yaptigi icin Gaussian Naive Bayes varsayimina daha uygundur.

In [ ]:
gender_df = df[["gender", "start_at", "duration_min", "distance_km_haversine"]].copy()
gender_df = gender_df[gender_df["gender"].isin([1, 2])].copy()
gender_df = gender_df[(gender_df["duration_min"] > 0) & (gender_df["distance_km_haversine"] > 0)].copy()

gender_df["speed_kmh"] = gender_df["distance_km_haversine"] / (gender_df["duration_min"] / 60)
gender_df = gender_df[(gender_df["speed_kmh"] >= 1) & (gender_df["speed_kmh"] <= 40)].copy()
gender_df["log_speed_kmh"] = np.log(gender_df["speed_kmh"])

gender_df = gender_df.sort_values("start_at").reset_index(drop=True)
display(gender_df.head())
display(gender_df.groupby("gender")[["speed_kmh", "log_speed_kmh"]].describe())

### Visualization ile dagilim secimi

Histogramlarda ham speed degiskeni saga kuyruklu gorunur. `log_speed_kmh` ise daha simetrik oldugu icin $P(log(speed) \mid gender)$ normal dagilim ile modellenir.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for gender_value, color, label in [(1, "#4c78a8", "Gender 1"), (2, "#e45756", "Gender 2")]:
    subset = gender_df.loc[gender_df["gender"] == gender_value]
    axes[0].hist(subset["speed_kmh"], bins=60, density=True, alpha=0.45, color=color, label=label)
    axes[1].hist(subset["log_speed_kmh"], bins=60, density=True, alpha=0.45, color=color, label=label)

axes[0].set_title("Speed distribution")
axes[0].set_xlabel("Speed, km/h")
axes[0].set_ylabel("Density")
axes[0].legend()

axes[1].set_title("Log speed distribution")
axes[1].set_xlabel("log(speed_kmh)")
axes[1].legend()

plt.tight_layout()
plt.show()

### Continuous Naive Bayes modeli

Model su sekilde kurulur:

$$P(gender=a \mid x) \propto P(gender=a) f_a(x)$$

Burada $x=log(speed)$ ve $f_a(x)$ gender sinifi icin normal dagilim yogunlugudur. Test seti yine zamana gore en sondaki %20 olarak ayrilir; boylece performans final test verisine bakarak optimize edilmez.

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix

nb_split_idx = int(len(gender_df) * 0.80)
nb_train = gender_df.iloc[:nb_split_idx].copy()
nb_test = gender_df.iloc[nb_split_idx:].copy()

class_stats = nb_train.groupby("gender")["log_speed_kmh"].agg(["mean", "std", "count"])
class_stats["prior"] = class_stats["count"] / class_stats["count"].sum()
class_stats["std"] = class_stats["std"].clip(lower=1e-6)
display(class_stats)

def gaussian_log_pdf(x, mean, std):
    return -np.log(std) - 0.5 * np.log(2 * np.pi) - 0.5 * ((x - mean) / std) ** 2

def predict_gender_from_speed(log_speed_values, stats):
    scores = pd.DataFrame(index=np.arange(len(log_speed_values)))
    for gender_value, row in stats.iterrows():
        scores[gender_value] = np.log(row["prior"]) + gaussian_log_pdf(
            log_speed_values.to_numpy(), row["mean"], row["std"]
        )
    return scores.idxmax(axis=1).astype(int).to_numpy()

nb_pred = predict_gender_from_speed(nb_test["log_speed_kmh"], class_stats)
nb_accuracy = accuracy_score(nb_test["gender"], nb_pred)
nb_confusion = pd.DataFrame(
    confusion_matrix(nb_test["gender"], nb_pred, labels=[1, 2]),
    index=["actual_1", "actual_2"],
    columns=["pred_1", "pred_2"],
)

print("Naive Bayes accuracy:", round(nb_accuracy, 4))
display(nb_confusion)

In [ ]:
x_grid = np.linspace(gender_df["log_speed_kmh"].quantile(0.01), gender_df["log_speed_kmh"].quantile(0.99), 300)

plt.figure(figsize=(8, 5))
for gender_value, color, label in [(1, "#4c78a8", "Gender 1"), (2, "#e45756", "Gender 2")]:
    subset = gender_df.loc[gender_df["gender"] == gender_value, "log_speed_kmh"]
    row = class_stats.loc[gender_value]
    density = np.exp(gaussian_log_pdf(x_grid, row["mean"], row["std"]))
    plt.hist(subset, bins=60, density=True, alpha=0.30, color=color, label=f"{label} real")
    plt.plot(x_grid, density, color=color, linewidth=2, label=f"{label} Gaussian model")

plt.title("Gaussian model for log speed by gender")
plt.xlabel("log(speed_kmh)")
plt.ylabel("Density")
plt.legend()
plt.show()